In [0]:
portfolio = spark.table("ifc.curated.portfolio")

print("Row count:", portfolio.count())
display(
    portfolio.select(
        "project_url",
        "client_id",
        "client_name",
        "internal_risk_rating",
        "performance_status",
        "total_approved_usd_mm",
        "disbursed_usd_mm"
    ).limit(10)
)

In [0]:
from importlib.metadata import version

for package in ["databricks-sdk", "databricks-connect", "openai"]:
    try:
        print(f"{package}: {version(package)}")
    except Exception:
        print(f"{package}: not installed")

portfolio = spark.table("ifc.curated.portfolio")
print("Portfolio rows:", portfolio.count())

In [0]:
from databricks.sdk import WorkspaceClient

workspace = WorkspaceClient()
MODEL = "system.ai.gpt-oss-120b"

response = workspace.api_client.do(
    method="POST",
    path="/ai-gateway/mlflow/v1/chat/completions",
    body={
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": "Reply with exactly: Model connected successfully."
            }
        ],
        "max_tokens": 1024
    }
)

print(response["choices"][0]["message"]["content"])

In [0]:
from pyspark.sql import functions as F

def get_client_profile(client_name: str) -> dict:
    portfolio = spark.table("ifc.curated.portfolio")

    result = (
        portfolio
        .filter(
            F.lower(F.trim(F.col("client_name")))
            == client_name.strip().lower()
        )
        .agg(
            F.countDistinct("project_url").alias("project_count"),
            F.sum("total_approved_usd_mm").alias("approved_usd_mm"),
            F.sum("disbursed_usd_mm").alias("disbursed_usd_mm"),
            F.sum(
                F.when(F.col("performance_status") == "Past Due", 1)
                 .otherwise(0)
            ).alias("past_due_projects"),
            F.sum(
                F.when(F.col("performance_status") == "Watchlist", 1)
                 .otherwise(0)
            ).alias("watchlist_projects"),
            F.sum(
                F.when(F.col("internal_risk_rating") >= 5, 1)
                 .otherwise(0)
            ).alias("high_risk_projects")
        )
        .first()
        .asDict()
    )

    if result["project_count"] == 0:
        return {"error": f"Client not found: {client_name}"}

    return {
        "client_name": client_name,
        "source_table": "ifc.curated.portfolio",
        "amount_unit": "USD millions",
        **result
    }

In [0]:
import json

TOOLS = [{
    "type": "function",
    "function": {
        "name": "get_client_profile",
        "description": (
            "Look up a client by exact name in ifc.curated.portfolio. "
            "Returns project count, approved and disbursed amounts in "
            "USD millions, and counts of Past Due, Watchlist and "
            "high-risk projects. High risk means internal rating >= 5."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "client_name": {
                    "type": "string",
                    "description": "Exact client name."
                }
            },
            "required": ["client_name"],
            "additionalProperties": False
        }
    }
}]

SYSTEM_PROMPT = """
You are IFC's Client Relationship Analyst.
Use get_client_profile before making factual claims about a client.
Base all figures on tool results. Amounts are in USD millions.
Distinguish approved investment from disbursed capital.
Do not treat missing values as zero.
Do not infer profitability or recommend investment from these
summary metrics alone. Explain what further evidence is needed.
If a client is not found, ask for the correct name.
Cite the source table: ifc.curated.portfolio.
"""

def ask_agent(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    for _ in range(5):
        response = workspace.api_client.do(
            method="POST",
            path="/ai-gateway/mlflow/v1/chat/completions",
            body={
                "model": MODEL,
                "messages": messages,
                "tools": TOOLS,
                "tool_choice": "auto",
                "max_tokens": 2048
            }
        )

        message = response["choices"][0]["message"]
        calls = message.get("tool_calls") or []

        if not calls:
            return message.get("content") or "No answer returned."

        messages.append({
            "role": "assistant",
            "content": message.get("content"),
            "tool_calls": calls
        })

        for call in calls:
            name = call["function"]["name"]
            arguments = json.loads(call["function"]["arguments"])

            print(f"Calling tool: {name}({arguments})")

            if name == "get_client_profile":
                result = get_client_profile(**arguments)
            elif name == "get_client_projects":
                result = get_client_projects(**arguments)
            else:
                result = {"error": f"Unknown tool: {name}"}

            print("Tool result:", result)

            messages.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "content": json.dumps(result, default=str)
            })

    raise RuntimeError("Agent exceeded the five-step tool-call limit.")

    TEST QUESTION


In [0]:
print(ask_agent(
    "Analyze Nirastek Energy PLC. Retrieve its summary and project-level "
    "evidence. Identify the high-risk project, its lifecycle status, "
    "approved and disbursed amounts, and days since last review. "
    "Include the project URL and data as-of date."
))

In [0]:
print(ask_agent(
    "Summarize Nirastek Energy PLC's portfolio and identify "
    "any risk signals in the available data."
))

In [0]:
# 避免重复添加
TOOLS = [
    tool for tool in TOOLS
    if tool["function"]["name"] != "get_client_projects"
]

TOOLS.append({
    "type": "function",
    "function": {
        "name": "get_client_projects",
        "description": (
            "Retrieve project-level evidence for an exact client name: "
            "project URL, project name, lifecycle status, environmental "
            "category, internal risk rating, performance status, approved "
            "and disbursed amounts in USD millions, days since review "
            "and data as-of date. Use this to investigate risk signals."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "client_name": {
                    "type": "string",
                    "description": "Exact client name."
                }
            },
            "required": ["client_name"],
            "additionalProperties": False
        }
    }
})

SYSTEM_PROMPT += """
When identifying a client's risk signals, retrieve project-level evidence
using get_client_projects in addition to the client summary.
Identify projects by name and URL, and report the data as-of date.
Distinguish closed projects from active exposure.
Approved amounts are not automatically signed commitments.
Do not calculate undisbursed commitments from approved amounts alone.
Do not infer overdue review status without an agreed review threshold.
"""

--------------------------Developing the first tool-------------------------------

Packaging Genie as a Python Tool - Query Portfolio (Question)

In [0]:

from databricks.sdk import WorkspaceClient

# Connect to the current Databricks workspace
w = WorkspaceClient()

# Replace with your existing Genie Agent ID
GENIE_SPACE_ID = "01f1d55aaa471bd7beae07107c391aef"

# Ask Genie a test question
question = "How many projects are in the IFC portfolio?"

response = w.genie.start_conversation_and_wait(
    space_id=GENIE_SPACE_ID,
    content=question
)

print("Status:", response.status)
print("Conversation ID:", response.conversation_id)

for attachment in response.attachments or []:
    if attachment.text:
        print("Answer:", attachment.text.content)
    if attachment.query:
        print("Generated SQL:", attachment.query.query)


In [0]:

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

GENIE_SPACE_ID = "01f1d55aaa471bd7beae07107c391aef"

def query_portfolio(question):
    """
    Ask Genie a financial question.
    Return generated SQL, text response, and actual query results.
    """
    message = w.genie.start_conversation_and_wait(
        space_id=GENIE_SPACE_ID,
        content=question
    )

    output = {
        "question": question,
        "status": str(message.status),
        "sql": [],
        "answers": [],
        "data": []
    }

    if str(message.status).split(".")[-1] != "COMPLETED":
        return output

    for attachment in message.attachments or []:

        if attachment.text:
            output["answers"].append(attachment.text.content)

        if attachment.query:
            output["sql"].append(attachment.query.query)

            if attachment.attachment_id:
                result = w.genie.get_message_attachment_query_result(
                    space_id=GENIE_SPACE_ID,
                    conversation_id=message.conversation_id,
                    message_id=message.message_id,
                    attachment_id=attachment.attachment_id
                )

                statement = result.statement_response

                if statement and statement.result:
                    output["data"].extend(
                        statement.result.data_array or []
                    )

    return output


Testing Genie Tool

In [0]:

result = query_portfolio(
    "How many projects are in the IFC portfolio?"
)

print("Status:", result["status"])
print("SQL:", result["sql"])
print("Data:", result["data"])
print("Answer:", result["answers"])


---------------------Developing the second tool - Risk Calculator (WS2)---------------------

According to WS2 in Case Brief, Expected Credit Loss = Disbursed Amount x 60%

In [0]:

def calculate_credit_risk():
    """
    Calculate credit exposure and expected losses
    for Category A projects on Watchlist or Past Due.
    """

    sql = """
    SELECT
        performance_status,
        COUNT(*) AS project_count,
        ROUND(
            SUM(COALESCE(disbursed_usd_mm, 0)), 2
        ) AS total_disbursed_usd_mm,
        ROUND(
            SUM(
                CASE
                    WHEN performance_status = 'Past Due'
                    THEN COALESCE(disbursed_usd_mm, 0) * 0.60
                    ELSE 0
                END
            ), 2
        ) AS expected_credit_loss_usd_mm
    FROM ifc.curated.portfolio
    WHERE environmental_category = 'A'
      AND performance_status IN ('Past Due', 'Watchlist')
      AND status <> 'Dropped'
    GROUP BY performance_status
    ORDER BY performance_status
    """

    return spark.sql(sql).toPandas()


In [0]:

risk_result = calculate_credit_risk()

display(risk_result)


Add an LLM as an Orchestrator - to decide which tools should be called/used by itself

In [0]:
import importlib.metadata as md

for package in ["databricks-sdk", "databricks-connect", "databricks-openai"]:
    try:
        print(package, md.version(package))
    except md.PackageNotFoundError:
        print(package, "Not installed")

print("Portfolio rows:", spark.table("ifc.curated.portfolio").count())

In [0]:

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

for endpoint in w.serving_endpoints.list():
    print(endpoint.name)


In [0]:
from databricks_openai import DatabricksOpenAI

client = DatabricksOpenAI()

Steps: 
1. Choosing databricks-gpt-oss-120b
2. Testing if LLM enables to answer questions

In [0]:

MODEL = "databricks-gpt-oss-120b"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are an AI financial analyst for IFC."
        },
        {
            "role": "user",
            "content": "What is Expected Credit Loss? Explain briefly."
        }
    ],
    max_tokens=300
)

print(response.choices[0].message.content)


3. Testing function calling

In [0]:

import json

test_tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate_credit_risk",
            "description": (
                "Calculate IFC Category A Past Due and "
                "Watchlist exposure and expected credit loss."
            ),
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": (
                "Calculate the expected credit loss "
                "for IFC Category A Past Due projects. "
                "Use the risk calculation tool."
            )
        }
    ],
    tools=test_tools,
    tool_choice="required"
)

message = response.choices[0].message

print("Tool calls:", message.tool_calls)

if message.tool_calls:
    print("Selected tool:", message.tool_calls[0].function.name)
    print("Arguments:", message.tool_calls[0].function.arguments)


4. Connecting with the Agent

In [0]:
query_portfolio(question)
calculate_credit_risk()

In [0]:

import json

def run_risk_agent(question):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a financial risk analyst for IFC. "
                "Use the available tool to calculate risk. "
                "Never invent financial figures. "
                "All amounts are in USD millions. "
                "Explain the calculations clearly."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    # 1. Ask the LLM to select a tool
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=test_tools,
        tool_choice="required"
    )

    assistant_message = response.choices[0].message
    messages.append(
        assistant_message.model_dump(exclude_none=True)
    )

    # 2. Actually execute the selected tool
    for call in assistant_message.tool_calls or []:

        print("Calling tool:", call.function.name)

        if call.function.name == "calculate_credit_risk":
            df = calculate_credit_risk()

            tool_result = df.to_dict(orient="records")

        else:
            tool_result = {
                "error": "Unknown tool"
            }

        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": json.dumps(
                tool_result,
                default=str
            )
        })

    # 3. Send the actual tool result back to the LLM
    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    return final_response.choices[0].message.content


5. Testing the whole completed Agent

In [0]:

answer = run_risk_agent(
    "Analyze IFC's Category A Past Due and Watchlist "
    "projects. How much disbursed capital is at risk? "
    "What is the expected credit loss? "
    "Explain the calculations."
)

print(answer)


6. Connecting Genie and Risk Calculator into the same agent

In [0]:

print("Genie tool:", callable(globals().get("query_portfolio")))
print("Risk tool:", callable(globals().get("calculate_credit_risk")))
print("LLM client:", "client" in globals())
print("Model:", globals().get("MODEL", "Not set"))


In [0]:

tools = [
    {
        "type": "function",
        "function": {
            "name": "query_portfolio",
            "description": (
                "Use Databricks Genie to answer questions about "
                "IFC portfolio investments, countries, regions, "
                "industries, projects, and client information."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": (
                            "The financial question to send to Genie."
                        )
                    }
                },
                "required": ["question"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_credit_risk",
            "description": (
                "Calculate Category A Past Due and Watchlist "
                "project counts, disbursed capital, and expected "
                "credit loss using the case-defined 60% loss "
                "assumption for Past Due projects."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "additionalProperties": False
            }
        }
    }
]

print("Registered tools:")
for tool in tools:
    print("-", tool["function"]["name"])


In [0]:
###*** This below is the original version
import json

SYSTEM_PROMPT = """
You are an AI financial analyst for IFC's CFO Office.

You have access to two tools:
1. query_portfolio: Queries IFC investment data using Databricks Genie.
2. calculate_credit_risk: Calculates Category A Past Due
   and Watchlist exposure and expected credit losses.

Rules:
- Use tools whenever financial data is needed.
- Never invent figures, client names, or calculations.
- All monetary amounts are in USD millions.
- The dataset is synthetic, not live IFC financial data.
- Use the case-defined 60% loss assumption only for
  Past Due projects in the specified risk calculation.
- A zero Watchlist ECL in this calculation does not
  imply that Watchlist projects have no credit risk.
- Distinguish approved amounts from disbursed capital.
- Explain assumptions and limitations.
- If a tool fails or its result is incomplete, say so.
- Do not make final investment recommendations
  without sufficient client-level evidence.
"""

def run_ifc_agent(question, max_steps=5):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    for step in range(max_steps):

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        assistant_msg = response.choices[0].message

        # No tool call: return the final response
        if not assistant_msg.tool_calls:
            return assistant_msg.content or "No response generated."

        # Preserve the assistant's tool call instructions
        messages.append(
            assistant_msg.model_dump(exclude_none=True)
        )

        # Execute every tool requested by the LLM
        for call in assistant_msg.tool_calls:

            tool_name = call.function.name

            print(f"Step {step + 1}: Calling {tool_name}")

            try:
                arguments = json.loads(
                    call.function.arguments or "{}"
                )

                if tool_name == "query_portfolio":
                    result = query_portfolio(
                        arguments["question"]
                    )

                    # Limit rows sent to the LLM
                    if isinstance(result, dict):
                        result["data"] = result.get(
                            "data", []
                        )[:20]

                elif tool_name == "calculate_credit_risk":
                    df = calculate_credit_risk()
                    result = df.to_dict(orient="records")

                elif tool_name == "analyze_client_risk":
                    result = client_risk_tool_handler(
                        client_id=arguments.get("client_id"),
                        limit=arguments.get("limit", 20)
                    )

                else:
                    result = {
                        "error": f"Unknown tool: {tool_name}"
                    }

                tool_output = json.dumps(
                    result, default=str
                )

            except Exception as e:
                tool_output = json.dumps({
                    "error": str(e),
                    "tool": tool_name
                })

            # Send the actual tool result back to the LLM
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": tool_output
            })

    return "Maximum tool steps reached without a final answer."
###***

In [0]:
### This is the newest version
def run_ifc_agent(question, max_steps=5):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    for step in range(max_steps):

        # 1. Let the LLM decide which tool to call
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        assistant_msg = response.choices[0].message

        # 2. If no tool is needed, return the final answer
        if not assistant_msg.tool_calls:
            return assistant_msg.content or "No response generated."

        # 3. Save the assistant's tool-call instructions
        messages.append(
            assistant_msg.model_dump(exclude_none=True)
        )

        # 4. Execute each requested tool
        for call in assistant_msg.tool_calls:

            tool_name = call.function.name

            print(f"Step {step + 1}: Calling {tool_name}")

            try:
                arguments = json.loads(
                    call.function.arguments or "{}"
                )

                # NEW: Execute tool through logging wrapper
                result = execute_agent_tool(
                    tool_name,
                    arguments
                )

                tool_output = json.dumps(
                    result,
                    default=str
                )

            except Exception as e:
                tool_output = json.dumps({
                    "status": "error",
                    "tool": tool_name,
                    "message": str(e)
                })

            # 5. Return tool results to the LLM
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": tool_output
            })

    return "Maximum tool steps reached without a final answer."


In [0]:

answer = run_ifc_agent(
    "First, tell me how many projects are in the "
    "IFC portfolio. Then calculate the total "
    "disbursed capital and expected credit loss "
    "for Category A Past Due and Watchlist projects."
)

print("\nFINAL ANSWER:\n")
print(answer)


7. Adding Financial Guardrails

In [0]:

SYSTEM_PROMPT = """
You are an AI financial analyst for IFC's CFO Office.

You analyze a synthetic IFC investment dataset using
Databricks Genie and validated financial calculation tools.

AVAILABLE TOOLS:
1. query_portfolio:
   Query portfolio investments, projects, countries,
   industries, clients and financial metrics.

2. calculate_credit_risk:
   Calculate credit exposure and expected credit loss
   for Environmental Category A projects with
   Past Due or Watchlist performance status.

CRITICAL BUSINESS DEFINITIONS:

- Environmental Category A refers to potentially
  significant environmental and social impacts.
  It is NOT a credit risk rating.

- Past Due and Watchlist are performance statuses,
  NOT environmental categories.

- Never confuse environmental risk with credit risk.

- Approved investment and disbursed capital
  are different financial measures.

- All financial amounts are in USD millions.

- Under the case-specific simplified ECL model:
  ECL = 60% * Past Due disbursed amount.

- Watchlist ECL is not estimated under this
  simplified model. A reported zero is NOT proof
  of zero expected losses.

BEHAVIOR RULES:

1. Always use tools for financial calculations.
2. Never invent numbers or client information.
3. Explain which filters were applied.
4. Clearly distinguish assumptions from facts.
5. Use only synthetic IFC dataset information.
6. If data is missing or incomplete, disclose it.
7. Never claim the dataset is live IFC internal data.
8. Show the evidence supporting financial conclusions.
9. Investment recommendations must be based on
   client-level evidence, not model intuition.
"""
print("Financial guardrails updated.")


In [0]:

answer = run_ifc_agent(
    "Explain the difference between Environmental "
    "Category A and Past Due / Watchlist status. "
    "Then calculate the credit exposure and ECL "
    "for Category A projects that are Past Due "
    "or on the Watchlist."
)

print(answer)


-----------------------------Developing the third tool---------------------------------

Client Analysis (WS3)

1. Checking the client data structure

In [0]:

client_check = spark.sql("""
SELECT
    client_id,
    client_name,
    COUNT(*) AS project_count,
    ROUND(SUM(total_approved_usd_mm), 2)
        AS total_approved_usd_mm,
    ROUND(SUM(COALESCE(disbursed_usd_mm, 0)), 2)
        AS total_disbursed_usd_mm,
    ROUND(AVG(internal_risk_rating), 2)
        AS avg_risk_rating,
    SUM(
        CASE WHEN performance_status = 'Past Due'
        THEN 1 ELSE 0 END
    ) AS past_due_count,
    SUM(
        CASE WHEN performance_status = 'Watchlist'
        THEN 1 ELSE 0 END
    ) AS watchlist_count
FROM ifc.curated.portfolio
WHERE client_id IS NOT NULL
  AND status <> 'Dropped'
GROUP BY client_id, client_name
ORDER BY total_approved_usd_mm DESC
LIMIT 10
""")

display(client_check)


2. Creating analyze_client_risk()

In [0]:

def analyze_client_risk(client_id=None, limit=20):
    """
    Analyze IFC clients using portfolio-level data.

    This is a preliminary analysis tool.
    Final risk scoring and investment recommendations
    will be added after WS3 SQL validation.

    All financial amounts are in USD millions.
    """

    if client_id is not None and not isinstance(client_id, str):
        raise ValueError("client_id must be a string")

    limit = int(limit)
    if not 1 <= limit <= 100:
        raise ValueError("limit must be between 1 and 100")

    df = spark.sql("""
        SELECT
            client_id,
            MAX(client_name) AS client_name,
            COUNT(*) AS project_count,

            ROUND(SUM(total_approved_usd_mm), 2)
                AS approved_usd_mm,

            ROUND(SUM(
                COALESCE(disbursed_usd_mm, 0)
            ), 2) AS disbursed_usd_mm,

            ROUND(AVG(internal_risk_rating), 2)
                AS avg_risk_rating,

            SUM(CASE
                WHEN performance_status = 'Past Due'
                THEN 1 ELSE 0
            END) AS past_due_count,

            SUM(CASE
                WHEN performance_status = 'Watchlist'
                THEN 1 ELSE 0
            END) AS watchlist_count,

            SUM(CASE
                WHEN performance_status = 'Performing'
                THEN 1 ELSE 0
            END) AS performing_count

        FROM ifc.curated.portfolio

        WHERE client_id IS NOT NULL
          AND status <> 'Dropped'

        GROUP BY client_id
    """)

    if client_id is not None:
        df = df.filter(df.client_id == client_id)

    from pyspark.sql import functions as F

    df = df.orderBy(
        F.col("approved_usd_mm").desc_nulls_last(),
        F.col("client_id")
    ).limit(limit)

    return df.toPandas()


3. Testing client analysis tool

In [0]:

client_result = analyze_client_risk(limit=10)

display(client_result)


In [0]:
client_result = analyze_client_risk(limit=10)
display(client_result)

In [0]:

# Register the third tool: Client Risk Analysis

client_risk_tool = {
    "type": "function",
    "function": {
        "name": "analyze_client_risk",
        "description": (
            "Analyze IFC clients using project-level "
            "investment and monitoring data. "
            "Returns client-level approved investment, "
            "disbursed capital, average internal risk rating, "
            "and counts of Performing, Past Due, and "
            "Watchlist projects. "
            "Use this tool for WS3 client analysis. "
            "It does NOT provide final investment rankings."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "client_id": {
                    "type": "string",
                    "description": (
                        "Optional client ID. Omit to analyze "
                        "clients across the portfolio."
                    )
                },
                "limit": {
                    "type": "integer",
                    "description": (
                        "Maximum number of clients to return, "
                        "between 1 and 100. Default is 20."
                    )
                }
            }
        }
    }
}

# Avoid duplicate registration when rerunning the cell
tools = [
    t for t in tools
    if t["function"]["name"] != "analyze_client_risk"
]

tools.append(client_risk_tool)

print("Registered tools:")
for tool in tools:
    print("-", tool["function"]["name"])


In [0]:

answer = run_ifc_agent(
    "Identify the 10 IFC clients with the highest "
    "approved investment. For each client, report "
    "their investment exposure, average internal "
    "risk rating, and number of Past Due and "
    "Watchlist projects. Do not make final "
    "investment recommendations yet."
)

print("\nFINAL ANSWER:\n")
print(answer)


---------------------------Testing Three Tools-------------------------

In [0]:

answer = run_ifc_agent(
    "Prepare a preliminary CFO briefing covering: "
    "1. Total number of IFC projects. "
    "2. Category A Past Due and Watchlist exposure "
    "and expected credit loss. "
    "3. The top 5 clients by approved investment "
    "and their risk indicators. "
    "Use the appropriate tools and explain "
    "the evidence. Do not recommend investments yet."
)

print(answer)


----------------------------Agent Evaluation & Validation--------------------------------

1. Basic Tool Validation

In [0]:
import pandas as pd

checks = []

def check(name, condition):
    passed = bool(condition)
    checks.append({
        "test": name,
        "status": "PASS" if passed else "FAIL"
    })
    print(("PASS" if passed else "FAIL"), "-", name)

# 1. Portfolio data integrity
portfolio = spark.table("ifc.curated.portfolio")

check(
    "Portfolio contains 5000 projects",
    portfolio.count() == 5000
)

check(
    "Project URLs are unique",
    portfolio.select("project_url").distinct().count()
    == portfolio.count()
)

# 2. Risk Calculator
risk = calculate_credit_risk()

check(
    "Risk Calculator returns 2 categories",
    set(risk["performance_status"])
    == {"Past Due", "Watchlist"}
)

past_due = risk[
    risk["performance_status"] == "Past Due"
].iloc[0]

check(
    "Past Due ECL follows 60% assumption",
    abs(
        float(past_due["expected_credit_loss_usd_mm"])
        - round(
            float(past_due["total_disbursed_usd_mm"]) * 0.60,
            2
        )
    ) <= 0.02
)

# 3. Client Analysis Tool
clients = analyze_client_risk(limit=10)

check(
    "Client Analysis returns results",
    not clients.empty
)

check(
    "Client IDs are unique in output",
    clients["client_id"].is_unique
)

check(
    "Approved investment is sorted descending",
    clients["approved_usd_mm"].is_monotonic_decreasing
)

# Summary
print("\nTEST SUMMARY")
display(pd.DataFrame(checks))
print(
    f"Passed: {sum(c['status'] == 'PASS' for c in checks)}"
    f"/{len(checks)}"
)


2. Agent Behavioral Evaluation

In [0]:
evaluation_questions = [
    {
        "id": "T1",
        "category": "Portfolio Analytics",
        "question": "How many projects are in the IFC portfolio?",
        "expected_tool": "query_portfolio"
    },
    {
        "id": "T2",
        "category": "Credit Risk",
        "question": (
            "Calculate the expected credit loss for "
            "Environmental Category A Past Due projects."
        ),
        "expected_tool": "calculate_credit_risk"
    },
    {
        "id": "T3",
        "category": "Client Analysis",
        "question": (
            "Identify the 10 clients with the highest "
            "approved investment and show their risk indicators. "
            "Do not make final investment recommendations."
        ),
        "expected_tool": "analyze_client_risk"
    },
    {
        "id": "T4",
        "category": "Multi-Tool",
        "question": (
            "Summarize total portfolio size, Category A "
            "credit exposure and the top 5 clients by "
            "approved investment."
        ),
        "expected_tool": "multiple"
    },
    {
        "id": "T5",
        "category": "Missing Information",
        "question": (
            "What will IFC's actual investment returns "
            "be in 2030? Provide exact figures."
        ),
        "expected_tool": "none_required"
    }
]

print("Evaluation questions prepared:",
      len(evaluation_questions))


Testing T1

In [0]:

test = evaluation_questions[0]

print("Test:", test["id"])
print("Question:", test["question"])
print("Expected tool:", test["expected_tool"])

answer = run_ifc_agent(test["question"])

print("\nAGENT RESPONSE:\n")
print(answer)


Testing T2: Credit Risk

In [0]:

test = evaluation_questions[1]

print("Test:", test["id"])
print("Question:", test["question"])
print("Expected tool:", test["expected_tool"])

answer = run_ifc_agent(test["question"])

print("\nAGENT RESPONSE:\n")
print(answer)


Testing T3: Client Analysis

In [0]:

test = evaluation_questions[2]

print("Test:", test["id"])
print("Question:", test["question"])
print("Expected tool:", test["expected_tool"])

answer = run_ifc_agent(test["question"])

print("\nAGENT RESPONSE:\n")
print(answer)


Testing T4: Multi-Tool Orchestration

In [0]:

test = evaluation_questions[3]

print("Test:", test["id"])
print("Question:", test["question"])
print("Expected tool:", test["expected_tool"])

answer = run_ifc_agent(test["question"])

print("\nAGENT RESPONSE:\n")
print(answer)


Add financial guardrails

In [0]:

SYSTEM_PROMPT += """

ADDITIONAL FINANCIAL ACCURACY RULES:

1. IFC internal risk ratings range from 1 to 6.
   1 = lowest risk, 6 = highest risk.
   Never describe the scale as 1 to 5.

2. Portfolio size can refer to project count or
   approved investment amount.
   When preparing a portfolio summary, report both
   if verified data is available.

3. Credit exposure and Expected Credit Loss (ECL)
   are different metrics.

4. For Category A Past Due and Watchlist projects:
   - Disbursed exposure is the sum of disbursed amounts.
   - ECL is calculated using the case-specific
     60% loss assumption for Past Due projects.
   - Do not describe ECL as total exposure.

5. Environmental Category A is an environmental
   and social classification, not a credit rating.

6. Never state that a risk metric is validated
   unless it has been checked against source data.
"""

print("Additional financial guardrails installed.")


In [0]:

answer = run_ifc_agent(
    "Prepare a concise CFO portfolio summary. "
    "Include total project count, total approved "
    "investment, Category A Past Due and Watchlist "
    "disbursed exposure, expected credit loss, "
    "and the top 5 clients by approved investment. "
    "Clearly distinguish exposure from expected loss."
)

print(answer)


Testing T5: Hallucination Test

In [0]:

test = evaluation_questions[4]

print("Test:", test["id"])
print("Question:", test["question"])
print("Expected tool:", test["expected_tool"])

answer = run_ifc_agent(test["question"])

print("\nAGENT RESPONSE:\n")
print(answer)


-----------------------Error Handling---------------------------

In [0]:

result = analyze_client_risk(
    client_id="CL-NOT-EXIST",
    limit=10
)

print("Returned rows:", len(result))

if result.empty:
    print("PASS: Client not found, no fabricated records.")
else:
    print("FAIL: Unexpected client records returned.")


In [0]:

def client_risk_tool_handler(client_id=None, limit=20):
    """
    Wrapper for analyze_client_risk.
    Provides structured tool responses for the Agent.
    """

    df = analyze_client_risk(
        client_id=client_id,
        limit=limit
    )

    if df.empty:
        return {
            "status": "not_found",
            "message": (
                f"No client records found for "
                f"client_id={client_id}"
                if client_id
                else "No client records found."
            ),
            "data": []
        }

    return {
        "status": "success",
        "row_count": len(df),
        "data": df.to_dict(orient="records"),
        "note": (
            "Results are preliminary client-level "
            "analytics, not validated WS3 recommendations."
        )
    }


In [0]:

print(client_risk_tool_handler(
    client_id="CL-NOT-EXIST"
))


In [0]:

answer = run_ifc_agent(
    "Analyze client CL-NOT-EXIST. "
    "Show its approved investment, "
    "risk rating and performance status."
)

print(answer)


-------------------Tool Call Logging & Observability------------------

In [0]:

import time
from datetime import datetime, timezone

agent_logs = []

def execute_agent_tool(tool_name, arguments):
    """
    Execute an agent tool and record execution metadata.
    """

    start = time.perf_counter()

    log = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "tool": tool_name,
        "arguments": arguments,
        "status": "started",
        "duration_seconds": None,
        "error": None
    }

    try:
        if tool_name == "query_portfolio":
            result = query_portfolio(arguments["question"])

            if isinstance(result, dict):
                result["data"] = result.get("data", [])[:20]

                if "COMPLETED" not in str(result.get("status", "")):
                    raise RuntimeError(
                        f"Genie status: {result.get('status')}"
                    )

        elif tool_name == "calculate_credit_risk":
            df = calculate_credit_risk()
            result = df.to_dict(orient="records")

        elif tool_name == "analyze_client_risk":
            result = client_risk_tool_handler(
                client_id=arguments.get("client_id"),
                limit=arguments.get("limit", 20)
            )

        elif tool_name == "recommend_clients":
            result = recommend_clients()
        
        else:
            raise ValueError(f"Unknown tool: {tool_name}")

        log["status"] = (
            result.get("status", "success")
            if isinstance(result, dict)
            else "success"
        )

    except Exception as e:
        log["status"] = "error"
        log["error"] = str(e)
        result = {
            "status": "error",
            "message": str(e)
        }

    finally:
        log["duration_seconds"] = round(
            time.perf_counter() - start, 3
        )
        agent_logs.append(log)

    return result


In [0]:

import pandas as pd

agent_logs.clear()

answer = run_ifc_agent(
    "Analyze client CL-NOT-EXIST and explain "
    "whether it has any investment exposure."
)

print("\nAGENT RESPONSE:\n")
print(answer)

print("\nTOOL EXECUTION LOGS:\n")
display(pd.DataFrame(agent_logs))


--------------------------------WS3 Integration Interface------------------------------

In [0]:

def recommend_clients():
    """
    WS3 recommendation interface.

    This tool will use the team's validated
    client scoring and investment recommendation
    SQL once available.

    Current status: pending integration.
    """

    return {
        "status": "not_ready",
        "workstream": "WS3",
        "message": (
            "Final client recommendation logic "
            "has not yet been integrated."
        ),
        "requirements": {
            "invest_more": 2,
            "reduce_or_review": 2,
            "include_undisbursed_commitments": True,
            "require_evidence": True
        }
    }

print(recommend_clients())


In [0]:

recommendation_tool = {
    "type": "function",
    "function": {
        "name": "recommend_clients",
        "description": (
            "Return WS3 investment recommendations "
            "for two clients to invest in and two "
            "clients to reduce or review. "
            "Currently unavailable until the team's "
            "validated recommendation model is integrated."
        ),
        "parameters": {
            "type": "object",
            "properties": {},
            "additionalProperties": False
        }
    }
}

tools = [
    t for t in tools
    if t["function"]["name"] != "recommend_clients"
]

tools.append(recommendation_tool)

print("Registered tools:")
for t in tools:
    print("-", t["function"]["name"])


In [0]:

answer = run_ifc_agent(
    "Recommend two IFC clients for additional "
    "investment and two clients whose exposure "
    "should be reduced or reviewed. "
    "Use the validated WS3 recommendation model."
)

print("\nAGENT RESPONSE:\n")
print(answer)
